# KVO_06 v2 — Tropical crown discovery and segmentation: Baego + Luki

**Clean, reproducible successor to** [the archived experimental NB06](https://github.com/kvanoost/Musanga/blob/main/KVO_06_SAM3_Baego_Crown_Discovery.ipynb).

### Hard data contract
- **Baego**: Kristof's *exhaustive* reference crowns in six evaluation ROIs; Nadine's precisely delineated but *sparse* crowns are an additional conditional boundary-quality reference. Do not interpret Nadine as exhaustive.
- **Luki**: photointerpretation crown polygons in `Tree_Annotation_Luki.gpkg`, layer `photointerpretation_inventory`; annotation-derived 10 m groups and 5 m buffers are *sampling areas*, **not confirmed exhaustive inventory plots**. Whole-stand precision, crown-count bias and F1 require separate annotation-completeness evidence.
- **Forbidden crown sites**: **Yoko, Yangambi, MOS_1_2**. Their crown annotations are partial and scattered across full mosaics. They must never enter this SAM crown notebook's training, validation, inference or crown metrics.
- **Musanga CNN**: read-only, frozen species probabilities (original CNN training sites unchanged). Do not confuse species probabilities with generic tree-crown validity.

### Section map
0. Environment and run controls → 1. Data harmonization → 2. Evaluation design → 3. Historic SAM1/SAM3 baseline → 4. Unified crown metrics → 5. SAM1/SAM2/SAM3 new inference adapters → 6. Spatial context experiments → 7. Selection and fusion → 8. Musanga integration → 9. Luki transfer → 10. Scientific reporting.

**Safe default:** read-only CPU analysis; no GPU inference or model training occurs unless explicitly enabled. Cached historical data are never overwritten. Results use `KVO_work/KVO_06_v2_Baego_Luki/`.


## 0 — Environment, Drive, paths and execution gates

Run this section at the beginning of **every new Colab runtime**. The default is `RUN_GPU_INFERENCE=False`, `RUN_SAM2=False`. Load only Baego and Luki crown inputs. The other three sites are absent from the site registry by design.


In [ ]:
import sys, json, hashlib, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import geopandas as gpd, rasterio
import matplotlib.pyplot as plt
from shapely.geometry import box, shape, mapping
from shapely.ops import unary_union
from shapely.validation import make_valid
from IPython.display import display
from google.colab import drive

ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
if not ROOT.exists(): drive.mount("/content/drive")
assert ROOT.exists(), "Drive mount or project root unavailable"
GEO=ROOT/"data"/"Geo_Ref"; ANNO=GEO/"Annotations"
OLD=ROOT/"KVO_work"/"KVO_06_UNet_guided_SAM3"
OUT=ROOT/"KVO_work"/"KVO_06_v2_Baego_Luki"
OUT.mkdir(parents=True,exist_ok=True)
PROB_DIR=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"

SITES={
    "Baego":{"rgb":GEO/"Baego_georef.tif",
             "kristof":ANNO/"Crown_Kristof.shp",
             "nadine":ANNO/"Crown_Nadine.shp",
             "evaluation_rois":ANNO/"ROI_Crowns_Kvo.shp",
             "probability":PROB_DIR/"Baego_musanga_probability_uint16.tif"},
    "Luki":{"rgb":GEO/"Luki.tif",
            "crowns":ANNO/"Tree_Annotation_Luki.gpkg",
            "layer":"photointerpretation_inventory",
            "probability":PROB_DIR/"Luki_musanga_probability_uint16.tif"}
}
assert set(SITES)=={"Baego","Luki"}
MIN_CROWN_M2=5.0
MATCH_IOU=.50
CORE_M=20.0
RUN_GPU_INFERENCE=False
RUN_SAM2=False
RUN_NEW_CROSS_SITE_INFERENCE=False
ALLOW_OVERWRITE=False
SEED=42
rng=np.random.default_rng(SEED)
print("KVO_06 v2 ready. Crown sites:",list(SITES))
print("All model execution gates OFF. Historic data protected.")


## 1 — Data inventory, annotation validity and completeness labels

Harmonizes crown polygons to each RGB raster CRS, preserving raw and filtered counts. The historical Baego `Crown_Kristof.shp` lacks its CRS metadata: following original NB06 Block14, only that file is assigned EPSG:32635 before reprojection (visually verify spatial alignment). Baego six-ROI crowns are **not** the same dataset as the independent Baego annotation used in NB04's ROI grouping. Never swap these references. Luki annotations can support crown-level conditional geometry even if exhaustive annotation coverage is uncertain.


In [ ]:
def polygons_only(g):
    if g is None or g.is_empty:return None
    g=make_valid(g)
    if g.geom_type in ("Polygon","MultiPolygon"):return g
    parts=[x for x in getattr(g,"geoms",[]) if x.geom_type in ("Polygon","MultiPolygon")]
    return unary_union(parts) if parts else None

def load_crowns(file,target_crs,layer=None,minimum_m2=MIN_CROWN_M2,verified_source_crs=None):
    kwargs={"layer":layer} if layer else {}
    frame=gpd.read_file(file,**kwargs)
    if frame.crs is None:
        if verified_source_crs is None:
            raise ValueError(f"{file} has no CRS. Provide a verified source CRS explicitly.")
        print(f"{Path(file).name}: missing .prj/CRS; assigning historically verified "
              f"source CRS {verified_source_crs} (check alignment visually).")
        frame=frame.set_crs(verified_source_crs)
    n0=len(frame)
    frame=frame[frame.geometry.notna() & ~frame.geometry.is_empty].copy()
    frame=frame.to_crs(target_crs)
    frame["geometry"]=frame.geometry.map(polygons_only)
    frame=frame[frame.geometry.notna() & ~frame.geometry.is_empty].copy()
    frame=frame[frame.geometry.area>0].copy()
    nvalid=len(frame)
    frame=frame[frame.geometry.area>=minimum_m2].reset_index(drop=True)
    return frame,{"input_n":n0,"valid_geometry_n":nvalid,
                  "excluded_lt_5m2":nvalid-len(frame),"retained_n":len(frame)}

DATA={};audit=[]
for name,s in SITES.items():
    assert s["rgb"].exists(),s["rgb"]
    with rasterio.open(s["rgb"]) as ds:
        assert ds.crs and ds.crs.is_projected,(name,"Need projected metric CRS")
        crs=ds.crs
        record={"site":name,"image_width":ds.width,"image_height":ds.height,
                "bands":ds.count,"crs":str(crs),"resolution_m":abs(ds.res[0])}
    if name=="Baego":
        ref,counts=load_crowns(s["kristof"],crs,minimum_m2=0.,verified_source_crs="EPSG:32635")
        nad,ncounts=load_crowns(s["nadine"],crs,minimum_m2=0.)
        rois=gpd.read_file(s["evaluation_rois"]).to_crs(crs)
        assert len(rois)==6,"Baego historical benchmark requires six ROIs"
        DATA[name]=dict(reference=ref,nadine=nad,rois=rois,crs=crs,
            completeness="exhaustive_within_six_Baego_ROIs")
        record.update(reference_count=len(ref),nadine_count=len(nad),
            evaluation_unit="six exhaustive polygon ROIs",
            annotation_completeness="exhaustive within six ROIs")
    else:
        ref,counts=load_crowns(s["crowns"],crs,layer=s["layer"])
        DATA[name]=dict(reference=ref,crs=crs,
            completeness="not_verified_as_exhaustive")
        record.update(reference_count=len(ref),nadine_count=0,
            evaluation_unit="annotated crown + derived spatial groups",
            annotation_completeness="NOT VERIFIED AS EXHAUSTIVE")
    record.update(counts)
    audit.append(record)
A=pd.DataFrame(audit)
display(A)
A.to_csv(OUT/"01_dataset_audit.csv",index=False)
print("DATA contract PASS: no Yoko, Yangambi or MOS crowns loaded.")


## 2 — Frozen evaluation protocol and spatial population

- **Autonomous crown discovery**: compare predictions against *exhaustive* Baego ROIs; standard precision/recall/F1, count and area bias, canopy IoU.
- **Conditioned geometry**: matching polygons against known Baego Nadine or Luki interpreted crowns supports positive-crown recovery / conditional IoU, **not** whole-stand precision.
- **Fixed 20 m cores**: historical Block70 provides 24 geometry-selected cores; reuse the saved manifest (do not resample).
- **Reference convention**: retain original Baego historical benchmark population for comparison, separately report ≥5 m²-filtered population. Do **not** silently apply the NB04 crown filter to old benchmark scores.
- **Splits, merges and incomplete reference annotations** are not automatically labelled as background errors.


In [ ]:
manifest=OLD/"experiments_v70_autonomous_sam3_multiscale"/"spatial_core_manifest.csv"
if manifest.exists():
    CORES=pd.read_csv(manifest)
    assert len(CORES)==24 and set(CORES.roi)==set(range(6))
    print("Historical Baego fixed-core manifest:",len(CORES),"cores")
    display(CORES.head())
else:
    CORES=None
    print("Historical fixed-core manifest missing: multispectral pilot unavailable until restored.")
print("Baego ROI polygons:",len(DATA["Baego"]["rois"]))
print("Luki crowns ≥5m²:",len(DATA["Luki"]["reference"]))
print("Whole-stand metrics eligibility: Baego six exhaustive ROIs only.")


## 3 — Reproduce historical SAM1/SAM3 benchmark, without rerunning inference

The authoritative old notebook is intentionally archived. Its Block64 summaries remain immutable, and are reproduced here **only from saved outputs**. These values are six-ROI *Baego development* results, not independent-site generalization.


In [ ]:
HIST=OLD/"experiments_v64_sixfold_musanga_gradient"
historic_metrics=HIST/"all6_frozen_metrics.csv"
if historic_metrics.exists():
    historic=pd.read_csv(historic_metrics)
    display(historic.head(20))
    print("Historical baseline loaded:",len(historic),"rows")
else:
    historic=None
    print("Block64 CSV absent. Do not regenerate old scores from changed labels.")


## 4 — Standard reusable crown-matching and ecological metrics

These functions can evaluate **fully annotated Baego ROIs** and also compute conditional geometry on Luki. A result must explicitly say which evaluation regime is used. One-to-one object matching uses **maximum cardinality** above the IoU threshold, not greedy pairing.


In [ ]:
from shapely.strtree import STRtree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

def crown_iou(g,h):
    if not g.intersects(h):return 0.
    a=g.intersection(h).area
    return float(a/max(g.area+h.area-a,1e-12))

def matched_objects(reference,predictions,threshold=MATCH_IOU):
    if len(reference)==0 or len(predictions)==0:return []
    tree=STRtree(predictions)
    edges=[]
    for i,g in enumerate(reference):
        for j in tree.query(g,predicate="intersects"):
            j=int(j)
            u=crown_iou(g,predictions[j])
            if u>=threshold:edges.append((i,j,u))
    if not edges:return []
    mat=csr_matrix((np.ones(len(edges),np.int8),
      ([e[0] for e in edges],[e[1] for e in edges])),
      shape=(len(reference),len(predictions)))
    matched=maximum_bipartite_matching(mat,perm_type="column")
    lookup={(i,j):v for i,j,v in edges}
    return [(i,int(j),lookup[i,int(j)]) for i,j in enumerate(matched) if j>=0]

def evaluate_crowns(reference,predictions,eval_polygon=None,
                    exhaustive=False,threshold=MATCH_IOU):
    refs=list(reference);preds=list(predictions)
    matches=matched_objects(refs,preds,threshold)
    result={"reference_n":len(refs),"predicted_n":len(preds),
            "matched_n":len(matches),"median_matched_iou":
            float(np.median([v for _,_,v in matches])) if matches else np.nan,
            "reference_exhaustive":bool(exhaustive)}
    if not exhaustive:
        # Sparse reference: no legitimate FP count or stand-level bias.
        result.update(conditional_recovery=len(matches)/len(refs) if refs else np.nan,
                      precision=np.nan,recall=np.nan,f1=np.nan,
                      count_bias_pct=np.nan,cpa_bias_pct=np.nan,canopy_iou=np.nan)
        return result
    tp=len(matches);fp=len(preds)-tp;fn=len(refs)-tp
    result.update(precision=tp/(tp+fp) if tp+fp else 0.,
        recall=tp/(tp+fn) if tp+fn else np.nan,
        f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan,
        count_bias_pct=100*(len(preds)-len(refs))/len(refs) if refs else np.nan)
    if eval_polygon is not None:
        rr=[g.intersection(eval_polygon) for g in refs]
        pp=[g.intersection(eval_polygon) for g in preds]
        ra=sum(g.area for g in rr);pa=sum(g.area for g in pp)
        ru=unary_union(rr) if rr else None;pu=unary_union(pp) if pp else None
        ar=ru.area if ru is not None else 0
        ap=pu.area if pu is not None else 0
        ix=ru.intersection(pu).area if ru is not None and pu is not None else 0
        result.update(cpa_bias_pct=100*(pa-ra)/ra if ra else np.nan,
            canopy_iou=ix/(ar+ap-ix) if ar+ap-ix else np.nan)
    return result
print("Metrics ready. Sparse annotations cannot yield stand-level precision.")


## 5 — SAM1 / SAM2 / SAM3 zero-shot automatic discovery: reproducible model adapters

All three models require **independent automatic-mask inference** to support a fair comparison. Model architectures, versions, weights, input scaling and candidate postprocessing must be recorded.

**SAM1**: original ViT-H automatic mask generator; existing Baego cache available.

**SAM2**: new official SAM2 automatic-mask generator; **not yet benchmarked** in the historic notebook. Requires compatible model checkpoint/configuration and Colab installation; never fabricate its scores.

**SAM3**: original frozen `facebook/sam3` Hugging Face mask-generation pipeline and existing Baego caches.

To avoid an accidental multi-hour run, the next cell is a manifest/configuration gate. It does not load checkpoints or run inference until `RUN_GPU_INFERENCE` is changed.


In [ ]:
MODEL_PROTOCOL={
    "SAM1_ViTH":{"family":"sam1","checkpoint":"external_original_vit_h_checkpoint",
                  "status":"historic_Baego_cache"},
    "SAM2_Hiera":{"family":"sam2","checkpoint":"official_SAM2_Hiera_checkpoint_required",
                  "status":"new_inference_pending"},
    "SAM3":{"family":"sam3","checkpoint":"facebook/sam3",
            "status":"historic_Baego_cache"}
}
MODEL_CONFIG=dict(min_area_m2=2.,max_area_m2=2500.,
                  match_iou=MATCH_IOU,core_m=CORE_M,
                  context_m=(40,80,160),
                  frozen_amg_pred_iou=.70,frozen_amg_stability=.80,
                  points_per_batch=16,points_per_crop=32)
signature=hashlib.sha256(json.dumps({
    "models":MODEL_PROTOCOL,"config":MODEL_CONFIG},
    sort_keys=True).encode()).hexdigest()[:12]
print("Experiment config hash:",signature)
display(pd.DataFrame([dict(model=k,**v) for k,v in MODEL_PROTOCOL.items()]))
if not RUN_GPU_INFERENCE:
    print("GPU inference intentionally DISABLED; existing caches remain unchanged.")
else:
    assert False, ("New GPU inference requires a separately reviewed, model-specific "
       "SAM1/SAM2/SAM3 adapter and verified checkpoint configurations. "
       "Do not silently reuse SAM3 settings for SAM2.")


## 6 — Spatial context: preserve the validated discovery insight

Reuses the historical Block70 40/80/160 m Baego output summaries. A proposal oracle is an **upper bound**, not operational precision or F1. Context resolution and sampled area must be equivalent across models before cross-model ranking.


In [ ]:
D70=OLD/"experiments_v70_autonomous_sam3_multiscale"
for filename in ("multiscale_pooled_summary.csv","multiscale_size_recovery.csv",
                 "paired_core_incremental_gain.csv"):
    f=D70/filename
    if f.exists():
        print("\n",filename)
        display(pd.read_csv(f).head(25))
    else:print("Not yet available:",f)


## 7 — Operational filtering, duplicate suppression and fusion

Reuses Blocks72–75 summaries **as scientific diagnostics**. Keep the baseline crown map fixed as the comparison. Existing tests showed that multiscale proposal recall improves but simple concatenation/NMS degrades F1 and biases ecological metrics. Do not present these as final independent validation.


In [ ]:
for block,folder,files in [
    ("72","experiments_v72_operational_multiscale_fusion",
     ["operational_pooled_summary.csv"]),
    ("73","experiments_v73_incremental_candidate_audit",
     ["operational_pooled_summary.csv","candidate_summary_by_scale.csv"]),
    ("74","experiments_v74_candidate_separability",
     ["candidate_outcomes.csv","cross_scale_consensus_summary.csv"]),
    ("75","experiments_v75_geometric_rgb_error_audit",
     ["error_category_summary.csv"])
]:
    for filename in files:
        f=OLD/folder/filename
        if f.exists():
            print("Block",block,filename)
            display(pd.read_csv(f).head(20))
        else:print("Missing archived diagnostic:",f)


## 8 — Species-specific Musanga CNN integration

Use the frozen Baego and Luki probability rasters for **species attribution to SAM crowns**, not as a general tree-validity filter. A Musanga probability raster must align in CRS, dimensions and transform with the relevant RGB mosaic. The output is only an uncalibrated crown-level CNN score until it has been independently validated.


In [ ]:
def audit_probability_alignment(site):
    spec=SITES[site];p=spec["probability"]
    if not p.exists():
        print(site,": probability map not yet available:",p)
        return False
    with rasterio.open(spec["rgb"]) as rgb,rasterio.open(p) as prob:
        ok=(rgb.crs==prob.crs and rgb.width==prob.width and
            rgb.height==prob.height and rgb.transform==prob.transform and
            prob.count==1 and prob.dtypes[0]=="uint16" and prob.nodata==65535)
        print(site,": frozen Musanga probability alignment", "PASS" if ok else "FAIL")
        assert ok,(site,"probability raster does not align with source RGB")
    return True
for site in SITES:audit_probability_alignment(site)
print("Species probability does not establish candidate crown authenticity.")


## 9 — Luki external transfer and visual reference QA

Luki's original crown annotations may use distinct digitization criteria. Do **not** infer whole-stand crown precision or count bias merely by buffering crown objects and calling the resulting regions exhaustive.

This section computes an annotation-derived spatial-group inventory (no GPU inference), and displays a selected original RGB window with correctly ordered Matplotlib extent (`left,right,bottom,top`). It is a **positive-crown geometry evaluation foundation**, not a final autonomous whole-stand benchmark.


In [ ]:
from shapely.strtree import STRtree
from rasterio.windows import from_bounds, bounds as window_bounds
luki=DATA["Luki"]["reference"]
geoms=list(luki.geometry)
parent=list(range(len(geoms)))
def find(i):
    while parent[i]!=i:
        parent[i]=parent[parent[i]];i=parent[i]
    return i
def join(i,j):
    a,b=find(i),find(j)
    if a!=b:parent[max(a,b)]=min(a,b)
tree=STRtree(geoms)
for i,g in enumerate(geoms):
    for j in tree.query(g.buffer(10)):
        j=int(j)
        if j>i and g.distance(geoms[j])<=10+1e-7:join(i,j)
groups={}
for i in range(len(geoms)):groups.setdefault(find(i),[]).append(i)
luki_groups=[]
for k,ids in enumerate(sorted(groups.values(),key=lambda x:-len(x)),1):
    geom=unary_union([geoms[j].buffer(5) for j in ids])
    luki_groups.append({"group_id":f"LUKI_{k:03d}",
                        "n_crowns":len(ids),"geometry":geom})
GROUPS_LUKI=gpd.GeoDataFrame(luki_groups,geometry="geometry",crs=luki.crs)
display(pd.DataFrame({"groups":[len(GROUPS_LUKI)],
                      "crowns":[len(luki)],"max_group":[GROUPS_LUKI.n_crowns.max()]}))
GROUPS_LUKI.to_file(OUT/"09_Luki_annotation_groups.gpkg",driver="GPKG")
# Correct geospatial extent order in imshow: (left, right, bottom, top)
with rasterio.open(SITES["Luki"]["rgb"]) as src:
    c=geoms[0].centroid;x,y=c.x,c.y
    w=from_bounds(x-15,y-15,x+15,y+15,src.transform).round_offsets().round_lengths()
    a=src.read([1,2,3],window=w)
    assert a.size and a.shape[1]>0 and a.shape[2]>0, "Empty Luki RGB window"
    left,bottom,right,top=window_bounds(w,src.transform)
fig,ax=plt.subplots(figsize=(7,7))
ax.imshow(np.moveaxis(a,0,-1),extent=(left,right,bottom,top),origin="upper")
for g in geoms:
    if g.intersects(box(left,bottom,right,top)):
        zz=[g] if g.geom_type=="Polygon" else list(g.geoms)
        for p in zz:
            xx,yy=p.exterior.xy;ax.plot(xx,yy,color="cyan",lw=1)
ax.set_xlim(left,right);ax.set_ylim(bottom,top);ax.set_aspect("equal")
ax.set_title("Luki RGB + original interpreted crowns (cyan)")
display(fig);plt.close(fig)


## 10 — Scientific status, reproducibility and next steps

This notebook currently supplies an **executable cleaned dataset/reference layer, frozen historical SAM1/SAM3 benchmark reader, standard metric functions, Luki reference/ROI QA and Musanga probability alignment QA**. It explicitly **does not claim to have generated fresh SAM1/SAM2/SAM3 multisite predictions or measured SAM2 performance**.

Next engineering tasks: (1) standardized three-model auto-mask adapters and frozen versioned cache contract; (2) sample Luki's independent RGB cores and determine annotation completeness; (3) run SAM2 on Baego and SAM1/SAM2/SAM3 on Luki; (4) separate conditional geometry results from exhaustive stand metrics; (5) controlled fusion and Musanga attribution; (6) final site-level reporting and uncertainty.

The original NB06 and its cached experiments remain the authoritative *history*, not a notebook to rerun in full.


In [ ]:
status={
  "notebook":"KVO_06_v2_Baego_Luki",
  "sites":["Baego","Luki"],
  "excluded_crown_sites":["MOS_1_2","Yoko","Yangambi"],
  "reference_policy":{"Baego":"Kristof exhaustive in six ROIs; Nadine sparse",
                      "Luki":"photointerpretation; exhaustiveness not verified"},
  "historic_baego_baseline_csv":str(historic_metrics),
  "sam2_benchmark":"pending independent inference",
  "gpu_inference_enabled":bool(RUN_GPU_INFERENCE),
  "experiment_protocol_hash":signature
}
f=OUT/"10_reproducibility_status.json"
f.write_text(json.dumps(status,indent=2))
print(json.dumps(status,indent=2))
print("KVO_06 v2 FOUNDATION COMPLETE. No training or SAM inference performed.")
